# Task 3: Agentic Financial Research with LangGraph
This notebook runs a single autonomous research agent and a two-agent analyst/writer workflow with mandatory clarification. Both reuse Task 1. The LLM chooses tools from observations; graph edges enforce the decision loop and role handoff, not a tool order.

Run all cells in order. Saved outputs reflect the last execution; rerun after pulling these fixes. No successful results are prefilled or fabricated. Real execution requires Yahoo/Groq/search availability and a Groq JSON-capable model. Do not paste keys into notebook source. Test demonstrations are separate offline pytest runs, not fabricated research results.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/Randimal/CDAZZDEV-MLE-Poshitha-Malagala.git"
cwd = Path.cwd()
repo_root = next(
    (
        p
        for p in (cwd, *cwd.parents)
        if (p / "task3_agentic" / "single_agent.py").exists()
    ),
    None,
)
if repo_root is None:
    if not REPOSITORY_URL.startswith("https://github.com/"):
        raise ValueError("Set REPOSITORY_URL and rerun this cell.")
    repo_root = cwd / "CDAZZDEV-MLE-Poshitha-Malagala"
    if not repo_root.exists():
        subprocess.run(
            ["git", "clone", "--", REPOSITORY_URL, str(repo_root)], check=True
        )
    if not (repo_root / "task3_agentic" / "single_agent.py").exists():
        raise ValueError("Checkout does not contain Task 3.")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(repo_root / "requirements.txt"),
    ],
    check=True,
)
sys.path.insert(0, str(repo_root))

## Configuration, secure credentials and free-tier settings
Load GROQ_API_KEY and GROQ_MODEL from environment, Colab Secrets or hidden input. The model is unchanged. Task3GroqClient uses request-specific completion caps: planner 768, critique 512, analyst brief 1200, clarification 900, sentiment 384, follow-up 640, final reports 1800. Brief/clarification stages need room for their typed finish outputs; writer_final retains the report cap because its next action is autonomous and may finish.

LOW reasoning effort is used only for verified supported SDK/model pairs (GPT-OSS 20B/120B); reports use MEDIUM. Other models receive no extra reasoning parameter. Set FREE_TIER_PACING=False for higher quotas. The default 60-second window is a conservative spacing policy: wait only the remainder since the last LLM completion between independent sections, not inside graph nodes. Provider Retry-After/backoff remains authoritative; pacing is not a guarantee against shared quotas or within-workflow TPM limits.

In [ ]:
import getpass
import logging
import os
from datetime import datetime, timezone

import pandas as pd
from IPython.display import JSON, display

from task1_financial.llm import LLMConfigurationError
from task3_agentic.demo_client import (
    CONTROLLED_DEMO_DECISIONS,
    DemoPacer,
    Task3GroqClient,
    TokenBudgets,
)
from task3_agentic.demonstrations import ControlledFailureExecutor
from task3_agentic.memory import PersistentMemory, answer_followup
from task3_agentic.multi_agent import TwoAgentResearch
from task3_agentic.prompts import QUESTION
from task3_agentic.runtime import AgentRuntime
from task3_agentic.schemas import ResearchReport
from task3_agentic.single_agent import SingleResearchAgent
from task3_agentic.tools import FinancialTools, ToolExecutor
from task3_agentic.tracing import ToolTracer

for name in ("GROQ_API_KEY", "GROQ_MODEL"):
    if not os.environ.get(name, "").strip():
        try:
            from google.colab import userdata

            value = userdata.get(name)
            if value:
                os.environ[name] = value.strip()
        except Exception:
            pass
if not os.environ.get("GROQ_API_KEY", "").strip():
    os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key (hidden): ").strip()
if not os.environ.get("GROQ_MODEL", "").strip():
    os.environ["GROQ_MODEL"] = input("Groq JSON-capable model identifier: ").strip()


logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
TICKER = "NVDA"
RUN_DATE = datetime.now(timezone.utc).date()
QUERY = QUESTION.format(ticker=TICKER)
TRACE_PATH = repo_root / "task3_agentic" / "logs" / "agent_trace.jsonl"
MEMORY_DIR = repo_root / "task3_agentic" / "memory"


# Disable pacing for higher-quota accounts. One minute is the TPM window,
# but the helper waits only its remaining part after the last completion.
FREE_TIER_PACING = True
PACE_WINDOW_SECONDS = 60.0
budgets = TokenBudgets()
pacer = DemoPacer(
    enabled=FREE_TIER_PACING, window_seconds=PACE_WINDOW_SECONDS, announce=print
)
print("Groq free-tier pacing enabled:", FREE_TIER_PACING)
display(vars(budgets))


client = None
try:
    client = Task3GroqClient(budgets=budgets)
except LLMConfigurationError:
    print("Configure GROQ_API_KEY and GROQ_MODEL, then rerun.")
except Exception:
    print("Client initialization unavailable; verify settings without sharing secrets.")


def make_runtime():
    tools = FinancialTools(TICKER, client)
    return AgentRuntime(client, ToolExecutor(tools, ToolTracer(TRACE_PATH)))


memory = PersistentMemory(MEMORY_DIR)


def pace_section(name):
    if client is not None:
        return pacer.before_section(name, client.last_completed_at)
    return 0.0

## Five callable tools — smoke tests only
This explicit showcase is separate from autonomous execution. Price/news/volatility/search still use the real production tools through the traced dispatcher. llm_sentiment analyses **ONE real retrieved headline** solely to prove it is callable; this is not the full market-sentiment assessment. The actual workflows remain free to select their own tools and perform full sentiment analysis. No result is faked. Yahoo/RSS source choice is logged.

In [ ]:
demo_observations = []
if client is not None:
    pace_section("the five-tool smoke showcase")
    demo = make_runtime().executor
    for name, arguments in (
        ("get_price_data", {"ticker": TICKER, "period": "2y"}),
        ("get_news", {"ticker": TICKER, "n": 10}),
        ("calculate_volatility", {"ticker": TICKER, "window": 60}),
        (
            "web_search",
            {"query": f"{TICKER} current share price risks official filings"},
        ),
    ):
        observation = demo.invoke("researcher", name, arguments)
        demo_observations.append(observation)
        print(name, "success:", observation.success)
        display(JSON(observation.model_dump(mode="json")))
    headlines = [vars(item) for item in demo.tools.known_headlines.values()][:1]
    print("llm_sentiment smoke test only — one real retrieved headline")
    observation = demo.invoke("researcher", "llm_sentiment", {"headlines": headlines})
    demo_observations.append(observation)
    display(JSON(observation.model_dump(mode="json")))
else:
    print("Configure Groq before the live tool showcase.")

## Single autonomous research and separate final synthesis
The LLM chooses tools, observes outcomes and replans without a fixed sequence.
Its finish action signals readiness, not a report. Code requires successful price/volatility AND news/search evidence before routing to the `final_synthesis` LangGraph node. The node receives a compact evidence digest and report schema, validates locally using Pydantic plus grounding checks, and allows at most one targeted repair. It cannot call tools or return to planning; failure produces no fabricated report.

Full OHLCV rows stay in state; planner context retains summary/latest indicators and evidence IDs. News contains titles/sources/dates and search has concise snippets/source hosts. Report schemas are absent from single-agent planning calls. Safe provider categories/status/codes and request character counts are visible; character counts are not measured tokens. Transport backoff stays in the existing Groq client.

In [ ]:
single = None
if client is not None:
    pace_section("the autonomous single-agent workflow")
    single_runtime = make_runtime()
    single = SingleResearchAgent(single_runtime, memory).run(
        QUERY, as_of=RUN_DATE, use_cache=False
    )
    print("Single-agent completed:", single.report is not None)
    if single.error:
        print("Controlled workflow failure:", single.error)

In [ ]:
def show_trace(run):
    display(
        pd.DataFrame(
            [
                {
                    "timestamp": item.timestamp,
                    "role": item.role,
                    "event": item.event,
                    "content": item.content,
                }
                for item in run.trace
            ]
        )
    )
    # Full sanitized messages with compact model-facing observations.
    requests = [item.content for item in run.trace if item.event == "llm_request"]
    display(
        pd.DataFrame(
            [
                {
                    "stage": item["user"]["stage"],
                    "prompt_chars": item.get("prompt_chars"),
                    "price_rows_sent": sum(
                        len((obs.get("output") or {}).get("rows", []))
                        for obs in item["user"]["observations"]
                    ),
                    "evidence_ids": item["user"]["allowed_evidence_ids"],
                }
                for item in requests
            ]
        )
    )
    display(JSON([item.model_dump(mode="json") for item in run.trace], expanded=True))


if single is not None:
    show_trace(single)
    cycle = [
        item.event
        for item in single.trace
        if item.event in {"decision", "tool_call", "observation", "replan"}
    ]
    print("Observed decision/tool/observation/replan cycle:", cycle)
    if single.report is not None:
        assert all(
            name in cycle for name in ("decision", "tool_call", "observation", "replan")
        )
        print(
            "Successful final report and autonomous observe/replan cycle demonstrated."
        )
    # Inspect the subsequent decision's reason to see how the model replanned.
    decisions = [item.content for item in single.trace if item.event == "decision"]
    display(JSON(decisions, expanded=True))
    print(
        "Separate final synthesis LLM requests:",
        sum(
            item.event == "llm_request"
            and item.content["user"]["stage"] == "final_synthesis"
            for item in single.trace
        ),
    )
    print("Financial Health Summary / Top Three Risks / Hedge Strategy Recommendation")
    display(
        JSON(
            single.report.model_dump(mode="json")
            if single.report
            else {"unavailable": single.error}
        )
    )

## Controlled failure injection for fallback demonstration — supplementary
ONE permitted tool actually selected by the LLM fails through the real dispatcher; the LLM chooses its next action after the observation. This is a controlled injection, not a real API outage. The demonstration uses the same graph with an existing configurable **three-decision budget**, rather than rerunning a full research/report. Budget exhaustion here is an intentional demonstration boundary, not a production setting. Inspect actual recovery/replan events; do not claim success when unavailable.

Previously fetched provider data can be reused through the single-agent FinancialTools session. No tool order or alternate action is fixed, and no production tool, permission, graph, memory/cache or synthesis contract changes. This supplementary cell does not regenerate a final report or save another research cache; the full single/multi reports are demonstrated separately. Tool-selected sentiment calls can still incur LLM cost.

In [ ]:
controlled = None
if client is not None:
    pace_section("the supplementary controlled-failure demo")
    # Reuse provider snapshots when available, without changing production tools.
    demo_tools = (
        single_runtime.executor.tools
        if single is not None
        else FinancialTools(TICKER, client)
    )
    injected = ControlledFailureExecutor(demo_tools, ToolTracer(TRACE_PATH))
    before_llm = client.call_count
    controlled = AgentRuntime(
        client, injected, max_decisions=CONTROLLED_DEMO_DECISIONS
    ).run(
        QUERY,
        role="researcher",
        stage="single_research",
        output_schema=ResearchReport,
        as_of=RUN_DATE,
    )
    print("Controlled failure injection for fallback demonstration")
    print("Injected tool selected by planner:", injected.injected_tool)
    print("Supplementary demo planning budget:", CONTROLLED_DEMO_DECISIONS)
    print(
        "Logical LLM calls including any selected sentiment tool:",
        client.call_count - before_llm,
    )
    show_trace(controlled)
    if injected.injected_tool is not None:
        assert controlled.observations and not controlled.observations[0].success
        print("Next autonomous decisions after failure:")
        display(
            JSON(
                [item.content for item in controlled.trace if item.event == "decision"][
                    1:
                ],
                expanded=True,
            )
        )
        print(
            "Successful alternative observations:",
            [item.evidence_id for item in controlled.observations[1:] if item.success],
        )
    else:
        print(
            "No tool was dispatched: the controlled failure has not yet been demonstrated."
        )
    print("Demo stopped:", controlled.error)
    print(
        "No duplicate final synthesis/report is requested by this supplementary three-decision demo."
    )

## Two complementary agents and mandatory useful critique
Agent A alone has price/volatility/sentiment tools; Agent B alone has news/search tools. A passes a validated Pydantic QuantitativeBrief (DataBrief). B retrieves qualitative evidence, then asks ONE specific missing quantitative or sentiment question that A can answer. A request for an already supplied non-null metric is rejected. The prompt prefers scoring B's retrieved headlines when sentiment is missing and useful; the question and metric are LLM-selected, not hard-coded. The actual accepted request and response appear below.

B's final report must copy A's answer into non-empty `clarification_used`, cite available clarification evidence, and acknowledge unassessed fundamentals. Volatility scaling is annualized_volatility * sqrt(90 / 252), under a 90-trading-day historical assumption. No optimal strike, attractive premium or cheap/expensive implied volatility can be asserted without option-chain data. Output attempts and transport backoff remain bounded.

In [ ]:
multi = None
if client is not None:
    pace_section("the two-agent workflow")
    multi_runtime = make_runtime()
    multi = TwoAgentResearch(multi_runtime, memory).run(
        QUERY, as_of=RUN_DATE, use_cache=False
    )
    print("Agent A quantitative brief")
    display(
        JSON(multi.brief.model_dump() if multi.brief else {"unavailable": multi.error})
    )
    print("Agent B qualitative research observations")
    display(
        JSON(
            [
                item.model_dump(mode="json")
                for item in multi.observations
                if item.role == "writer"
            ],
            expanded=True,
        )
    )
    print("Agent B critique request")
    display(
        JSON(
            multi.critique_request.model_dump()
            if multi.critique_request
            else {"unavailable": True}
        )
    )
    print("Agent A clarification response")
    display(
        JSON(
            multi.clarification.model_dump()
            if multi.clarification
            else {"unavailable": True}
        )
    )
    print("Agent B final report")
    display(
        JSON(
            multi.report.model_dump(mode="json")
            if multi.report
            else {"unavailable": multi.error}
        )
    )
    show_trace(multi)
    if multi.report is not None:
        assert multi.critique_request is not None and multi.clarification is not None
        assert (
            multi.report.clarification_used and multi.report.clarification_used.strip()
        )
        assert multi.report.clarification_used == multi.clarification.answer
        supplied_metrics = {
            metric.name for metric in multi.brief.metrics if metric.value is not None
        }
        assert multi.critique_request.requested_metric not in supplied_metrics
        print("Missing analysis requested:", multi.critique_request.requested_metric)
        print("Accepted clarification_used:", multi.report.clarification_used)
        print("Mandatory critique loop completed and incorporated.")

## Forbidden-tool access is blocked by the dispatcher
These deliberately invalid role requests use the production permission checks. They are controlled failures, not external API calls. Agent A cannot search; Agent B cannot fetch price data. Denied attempts are logged and displayed as failed ToolObservation records.

In [ ]:
if client is not None:
    restricted = make_runtime().executor
    before_llm = client.call_count
    for role, name, arguments in (
        ("analyst", "web_search", {"query": f"{TICKER} risk"}),
        ("writer", "get_price_data", {"ticker": TICKER}),
    ):
        denied = restricted.invoke(role, name, arguments)
        display(JSON(denied.model_dump(mode="json")))
        assert not denied.success and "PermissionError" in denied.error
        print(role, "->", name, "blocked by runtime permissions")
    assert client.call_count == before_llm
    assert not restricted.tools._prices

## Short-term memory: no repeat price/volatility fetch
The follow-up helper receives saved state and has no access to a tool executor. It may make an LLM call, but cannot refetch data. Tool-call counters and trace-file size verify that property.

In [ ]:
if multi is not None and multi.report is not None:
    pace_section("the short-term memory follow-up")
    before = multi_runtime.executor.counter
    trace_bytes = TRACE_PATH.stat().st_size if TRACE_PATH.exists() else 0
    followup = answer_followup(
        multi,
        "What quantitative or sentiment clarification did Agent A supply, and what limitations remain?",
        client,
    )
    after = multi_runtime.executor.counter
    display(JSON(followup.model_dump() if followup else {"unavailable": True}))
    display(
        {
            "tool_calls_before": before,
            "tool_calls_after": after,
            "new_tool_calls": after - before,
        }
    )
    assert after == before
    assert TRACE_PATH.stat().st_size == trace_bytes
    if followup is not None:
        print("Follow-up answered from session state with zero new tool calls.")
    else:
        print(
            "Follow-up LLM unavailable; memory did not refetch data. Rerun after checking provider availability."
        )
else:
    print("A completed workflow is required for the follow-up demonstration.")

## Persistent memory: first saved run, then a new-session cache hit
A valid report from the first run is saved to TICKER_UTC-DATE.json. Single and multi results occupy separate workflow slots so a cached single report cannot bypass the required two-agent critique. Cache bypass is explicit with use_cache=False. A new client-free session below proves the second run does not require API calls.

In [ ]:
if multi is not None and multi.report is not None:
    cache_path = memory.path(TICKER, RUN_DATE)
    print(
        "First run saved:",
        cache_path.relative_to(repo_root),
        "exists:",
        cache_path.exists(),
    )

    class NoCallsClient:
        calls = 0

        def complete(self, system, user):
            self.calls += 1
            raise AssertionError("A cache hit must not call an LLM")

    cached_client = NoCallsClient()
    cached_runtime = AgentRuntime(
        cached_client,
        ToolExecutor(FinancialTools(TICKER, cached_client), ToolTracer(TRACE_PATH)),
    )
    assert cache_path.exists(), "First-run cache save did not complete."
    before_bytes = TRACE_PATH.stat().st_size
    second = TwoAgentResearch(cached_runtime, memory).run(QUERY, as_of=RUN_DATE)
    assert second.cached and cached_runtime.executor.counter == 0
    assert TRACE_PATH.stat().st_size == before_bytes
    proof = {
        "cache_hit": second.cached,
        "new_tool_calls": cached_runtime.executor.counter,
        "new_llm_calls": cached_client.calls,
    }
    display(proof)
    assert proof == {"cache_hit": True, "new_tool_calls": 0, "new_llm_calls": 0}
    print("Second run: persistent cache hit; zero tools and zero LLM calls.")
    display(JSON(second.report.model_dump(mode="json")))
    display(JSON(second.trace[-1].model_dump(mode="json")))
else:
    print(
        "No completed report: nothing was cached. Retry after resolving availability."
    )

## Tool observability
Every dispatcher invocation, including permission failures and session cache hits, appends a redacted record. Output is limited to 200 characters; full model-facing observations remain in the session trace. No API keys or raw exception bodies are logged. The required task3_agentic/logs/agent_trace.jsonl is intentionally committed; temporary logs and cache files remain ignored. Review the real trace for secrets before committing.

In [ ]:
import json

if TRACE_PATH.exists():
    rows = [
        json.loads(line) for line in TRACE_PATH.read_text(encoding="utf-8").splitlines()
    ]
    display(pd.DataFrame(rows).tail(15))
    assert all(len(row["output"]) <= 200 for row in rows)
    print("JSONL tool-call count:", len(rows))
else:
    print("No tool trace yet; execute the live workflow cells first.")

## Limitations and submission checks
Run the updated notebook in a fresh Colab runtime after publishing/pulling the changed Task 3 files. Execute all cells in order. The main first runs explicitly bypass persistent cache; the later same-day cache demonstration must show a true hit with zero new tool/LLM calls. Inspect actual outputs for the single report, decision/tool/observation/replan cycle, injected failure and autonomous response, A's brief, B's research/missing-analysis critique, A's response, and B's incorporated final report. Do not claim these live demonstrations succeeded unless their real outputs show success.

Provider availability/free-tier quotas may interrupt execution. The supplementary controlled demo is limited to three planning decisions and no duplicate report. Free-tier pacing can be disabled; it spaces independent sections but cannot eliminate within-workflow or shared-account TPM limits. Smaller completion caps are configurable and still subject to provider output truncation; report caps remain 1800. Transport backoff is bounded; synthesis allows one targeted repair and never fabricates a report. Search snippets/headlines may be stale, incomplete or misleading. Evidence/reference checks do not prove all causal claims. OHLCV cannot establish audited financial health. Historical volatility assumes a window/stationarity convention; 90 trading days differs from 90 calendar days. Hedge concepts require option quotes/IV/liquidity and portfolio review before execution. Same-day cache can be stale: use_cache=False refreshes it. Review the required commit-able trace for secrets. Task 1 remains frozen and Task 2 pending.

After updating the checkout, restart Colab and run all cells once with FREE_TIER_PACING=True. Let the announced waits complete. If a rate limit still exhausts transport retries, wait for the provider reset and rerun only the affected workflow cell and its dependent display/memory/cache cells; avoid repeating an already successful smoke showcase. Saved outputs have not been cleared or fabricated by this update.